<a href="https://colab.research.google.com/github/sathkumara-smna/258739K/blob/main/Site_Power_Prection_Eng_Rule.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

site_db_url = "https://raw.githubusercontent.com/sathkumara-smna/258739K/main/site_power%20from%20Sey.xlsx"
site_db = pd.read_excel(site_db_url)
site_db.head(2)

In [ ]:
power_2g3g_url = "https://raw.githubusercontent.com/sathkumara-smna/258739K/main/site_power_2g3g.xlsx"
power_2g3g = pd.read_excel(power_2g3g_url)
power_2g3g.head(2)

In [ ]:
power_4g_url = "https://raw.githubusercontent.com/sathkumara-smna/258739K/main/site_power_lte.xlsx"
power_4g = pd.read_excel(power_4g_url)
power_4g.head(2)

In [ ]:
power_5g_url = "https://raw.githubusercontent.com/sathkumara-smna/258739K/main/site_power_5g.xlsx"
power_5g = pd.read_excel(power_5g_url)
power_5g.head(2)

In [ ]:
# Add power_2g3g column to site_db using Site_ID matching

site_db['power_2g3g'] = (

    site_db['Site_ID'].map(power_2g3g.set_index('Site_ID')['total_2g_3g_power'])
)
# Show first rows
site_db.head()

In [ ]:
# Add 4G power column to site_db
# Matching using Site_ID, trigger_ID, and date

site_db = site_db.merge(

    power_4g[
        ['Site_ID', 'trigger_ID', 'date', 'site_power']
    ].rename(columns={'site_power': 'power_4g'}),

    on=['Site_ID', 'trigger_ID', 'date'],how='left')

# Show first rows
site_db.head(2)

In [ ]:
# Add 5G power column to site_db
# Matching using Site_ID, trigger_ID, and date

site_db = site_db.merge(

    power_5g[
        ['Site_ID', 'trigger_ID', 'date', 'site_5g_power']
    ].rename(columns={'site_5g_power': 'power_5g'}
    ),
    on=['Site_ID', 'trigger_ID', 'date'],
    how='left'

)
# Show first rows
site_db.head()

In [ ]:
# Replace empty values with 0

site_db['power_2g3g'] = (site_db['power_2g3g'].fillna(0))
site_db['power_4g'] = (site_db['power_4g'].fillna(0))
site_db['power_5g'] = (site_db['power_5g'].fillna(0))

# Recalculate total predicted power
site_db['site_total_predicted_power'] = (site_db['power_2g3g']+site_db['power_4g']+site_db['power_5g'])
# Show first rows
site_db.head(2)

In [ ]:
# ============================================================
# ERROR COLUMN
# ============================================================
site_db['error'] = (site_db['site_power']-site_db['site_total_predicted_power'])
# ============================================================
# SHOW RESULTS
# ============================================================

site_db.head(2)

In [ ]:
# ============================================================
# ERROR PERCENTAGE COLUMN
# ============================================================

site_db['error_percentage'] = (

    site_db['error']

    /

    site_db['site_power']

) * 100

# ============================================================
# SHOW RESULTS
# ============================================================

site_db.head()

In [ ]:
site_db.to_excel('Total_Power_Prediction_Eng_Rule.xlsx',index=False)

In [ ]:
# ============================================================
# ERROR PERCENTAGE
# ============================================================

site_db['error_percentage'] = (
    np.abs(site_db['site_power'] - site_db['site_total_predicted_power'])/site_db['site_power']) * 100

# ============================================================
# MAXIMUM AND MINIMUM ERROR %
# ============================================================

max_error = site_db['error_percentage'].max()
min_error = site_db['error_percentage'].min()

# ============================================================
# PRINT RESULTS
# ============================================================

print("================================")
print("ENGINEERING RULE PERFORMANCE")
print("================================")

print(f"MAE               : {round(mae, 2)}")
print(f"RMSE              : {round(rmse, 2)}")
print(f"MAPE              : {round(mape, 2)} %")
print(f"R2                : {round(r2, 4)}")

print("--------------------------------")

print(f"Maximum Error %   : {round(max_error, 2)} %")
print(f"Minimum Error %   : {round(min_error, 2)} %")

In [ ]:
# ============================================================
# IMPORT LIBRARIES
# ============================================================

import matplotlib.pyplot as plt

# ============================================================
# ACTUAL VS PREDICTED POWER GRAPH
# ============================================================

plt.figure(figsize=(16,6))

plt.plot(
    site_db['datetime'],
    site_db['site_power'],
    label='Actual Site Power'
)

plt.plot(
    site_db['datetime'],
    site_db['site_total_predicted_power'],
    label='Predicted Site Power'
)

plt.xlabel('Datetime')
plt.ylabel('Power (W)')
plt.title('Actual vs Predicted Site Power')
plt.legend()
plt.grid(True)
plt.show()

# ============================================================
# SCATTER PLOT
# ============================================================

plt.figure(figsize=(8,8))

plt.scatter(
    site_db['site_power'],
    site_db['site_total_predicted_power']
)

plt.xlabel('Actual Site Power')
plt.ylabel('Predicted Site Power')
plt.title('Actual vs Predicted Scatter Plot')
plt.grid(True)
plt.show()

# ============================================================
# ERROR PERCENTAGE HISTOGRAM
# ============================================================

plt.figure(figsize=(10,6))

plt.hist(
    site_db['error_percentage'],
    bins=30
)

plt.xlabel('Error Percentage (%)')
plt.ylabel('Frequency')
plt.title('Prediction Error Distribution')
plt.grid(True)
plt.show()

# ============================================================
# SITE-WISE AVERAGE ERROR
# ============================================================

site_error = (

    site_db.groupby('Site_ID')['error_percentage']
    .mean()
    .reset_index()

)

site_error = site_error.sort_values(
    by='error_percentage',
    ascending=False
)

plt.figure(figsize=(16,6))

plt.bar(
    site_error['Site_ID'].astype(str),
    site_error['error_percentage']
)

plt.xticks(rotation=90)
plt.xlabel('Site_ID')
plt.ylabel('Average Error %')
plt.title('Site-wise Average Prediction Error')
plt.grid(True)
plt.show()

# ============================================================
# TECHNOLOGY CONTRIBUTION PIE CHART
# ============================================================

avg_2g3g = site_db['power_2g3g'].mean()
avg_4g = site_db['power_4g'].mean()
avg_5g = site_db['power_5g'].mean()
plt.figure(figsize=(8,8))
plt.pie(

    [
        avg_2g3g,
        avg_4g,
        avg_5g
    ],

    labels=[
        '2G + 3G',
        '4G',
        '5G'
    ],

    autopct='%1.1f%%'

)

plt.title('Average Technology-wise Power Contribution')

plt.show()